# Rejection Sampling vs. Policy Gradient

## 1. Core Intuition

**Rejection Sampling + SFT is like manually performing Policy Gradient.**

Both methods use reward signals to increase the probability of generating good answers.

The key difference is **how reward information enters the gradient update**:

- **Rejection Sampling + SFT:** Use rewards to select good samples, then apply standard SFT to increase their probabilities.
- **Policy Gradient:** Use rewards directly to weight gradient updates.

Importantly, Rejection Sampling does not manually perform the gradient calculation. SFT still uses backpropagation. The manual part is deciding **which samples deserve a gradient update**.

## 2. A Minimal Example: 30% Accuracy

Suppose a model can generate only two answers:

| Answer | Reward | Probability |
|---|---|---|
| A (correct) | 1 | 0.3 |
| B (incorrect) | 0 | 0.7 |

Let the model have a single parameter $\theta$, with:

$$
p=P(A)=\sigma(\theta)
$$

where $\sigma$ is the sigmoid function.

Our goal is to increase $p$, the probability of generating the correct answer.

### Method A: Rejection Sampling + SFT

We repeatedly sample until we find the correct answer A, then use A as the SFT training target.

SFT maximizes the log probability of A:

$$
J_{\text{SFT}}(\theta)=\log p
$$

Since:

$$
\frac{dp}{d\theta}=p(1-p)
$$

the gradient is:

$$
\begin{aligned}
\nabla_\theta J_{\text{SFT}}
&=\frac{1}{p}\frac{dp}{d\theta}\\
&=\frac{1}{p}p(1-p)\\
&=1-p
\end{aligned}
$$

At $p=0.3$:

$$
\boxed{\nabla_\theta J_{\text{SFT}}=0.7}
$$

The gradient is positive, so gradient ascent increases $\theta$, which increases $P(A)$.

### Method B: Policy Gradient

RL directly maximizes expected reward:

$$
J_{\text{RL}}(\theta)
=\mathbb{E}_{y\sim\pi_\theta}[R(y)]
$$

Since A receives reward 1 and B receives reward 0:

$$
\begin{aligned}
J_{\text{RL}}(\theta)
&=P(A)\cdot1+P(B)\cdot0\\
&=p
\end{aligned}
$$

Therefore:

$$
\nabla_\theta J_{\text{RL}}
=\frac{dp}{d\theta}
=p(1-p)
$$

At $p=0.3$:

$$
\boxed{\nabla_\theta J_{\text{RL}}=0.21}
$$

### Comparing the Gradients

| | RS + SFT | Policy Gradient |
|---|---|---|
| Objective | $\log p$ | $p$ |
| Gradient | $1-p$ | $p(1-p)$ |
| At $p=0.3$ | $+0.7$ | $+0.21$ |
| Direction | Positive | Positive |

Notice:

$$
\boxed{
\nabla_\theta J_{\text{RL}}
=p\,\nabla_\theta J_{\text{SFT}}
}
$$

Since $p>0$, both gradients point in exactly the same direction.

**Key insight: In this simple binary example, Rejection Sampling + SFT and Policy Gradient produce the same gradient direction, differing only by a positive scaling factor.**

This does not imply that the two methods are generally equivalent. The result relies on the simplified binary setup, fixed rewards, and a single model parameter.

## 3. Why Are They So Similar?

The likelihood-ratio (REINFORCE) identity gives:

$$
\nabla_\theta J_{\text{RL}}
=
\mathbb{E}_{y\sim\pi_\theta}
\left[
R(y)\nabla_\theta\log\pi_\theta(y)
\right]
$$

There are two components:

1. $R(y)$: How good is this answer?
2. $\nabla_\theta\log\pi_\theta(y)$: How do we make this answer more likely?

Policy Gradient combines these two components directly.

Rejection Sampling + SFT separates them into two steps:

1. Use $R(y)$ to select good answers.
2. Use $\nabla_\theta\log\pi_\theta(y)$ to increase their probabilities.

**Both methods use reward signals to reinforce desirable behavior, but they incorporate those signals differently.**

## 4. The Fundamental Difference: Reward Information

Rejection Sampling typically converts reward scores into selection decisions.

Consider four candidate answers:

| Answer | Reward | RS (Top-1) | Policy Gradient |
|---|---|---|---|
| A | 0.95 | Selected | Reward 0.95 |
| B | 0.90 | Discarded | Reward 0.90 |
| C | 0.40 | Discarded | Reward 0.40 |
| D | 0.10 | Discarded | Reward 0.10 |

For Top-1 Rejection Sampling, the selection decision effectively converts continuous scores into binary labels:

$$
R(y)\longrightarrow
\begin{cases}
1 & \text{if selected}\\
0 & \text{otherwise}
\end{cases}
$$

The selected answer becomes a supervised learning target.

Policy Gradient, by contrast, can retain information about reward magnitude when computing updates.

However, Policy Gradient does not simply increase each answer's probability in proportion to its raw reward. The actual update depends on the policy probabilities, reward values, and potentially a baseline or advantage function.

**Key trade-off:** Rejection Sampling offers a simple training pipeline but discards some reward information. Policy Gradient can use richer reward signals but introduces additional optimization challenges.

## 5. Another Important Difference: Training Objective

The two approaches optimize different objectives.

### Rejection Sampling + SFT

$$
\max_\theta\;
\mathbb{E}_{(x,y^*)\sim D_{\text{selected}}}
[\log\pi_\theta(y^*\mid x)]
$$

The model learns to imitate selected high-quality answers.

### Policy Gradient

$$
\max_\theta\;
\mathbb{E}_{x,\;y\sim\pi_\theta(\cdot\mid x)}
[R(x,y)]
$$

The model directly optimizes expected reward.

This difference has important consequences:

- RS + SFT can repeatedly train on a fixed dataset of selected answers.
- On-policy RL typically collects new samples as the policy changes.
- RS optimizes imitation of selected outputs; RL optimizes expected reward under the policy.
- Their gradient directions can coincide in simple cases but differ in general.

## 6. The Bigger Picture

Rejection Sampling + SFT can be summarized as:

**Generate → Evaluate → Select → Imitate**

Policy Gradient can be summarized as:

**Generate → Evaluate → Reward-weighted Gradient Update**

Both ultimately modify model parameters through gradient-based optimization.

The distinction is not *gradient descent versus no gradient descent*. It is **how the reward signal is converted into a learning signal**.

## 7. Open Questions Worth Exploring

1. **When are RS + SFT and Policy Gradient mathematically equivalent?** Beyond a binary example, what assumptions are necessary for their gradients to align?

2. **Why do we need PPO or GRPO if RS + SFT already works?** What advantages come from directly optimizing expected reward rather than imitating selected answers?

3. **Can a model continuously improve by training on its own best outputs?** What limits self-improvement when the model generates its own training data?

4. **What happens when the Reward Model is imperfect?** Can repeatedly selecting high-reward answers amplify reward-model errors?

## Final Takeaway

> **Rejection Sampling + SFT and Policy Gradient are two ways of turning reward information into higher probabilities for desirable outputs.**
>
> Rejection Sampling uses rewards to decide *what to imitate*. Policy Gradient uses rewards to determine *how to update the policy*.
>
> In a simple binary-reward example, their gradients point in exactly the same direction. The deeper differences emerge when we consider multiple actions, non-binary rewards, changing policies, and repeated optimization.